# Lambert Liu Runner

In [1]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_hyper_tuning import Tuner
import c_clustering as c
import numpy as np
import polars as pl
from numba import set_num_threads, get_num_threads

set_num_threads(15)
get_num_threads()

15

### Loading in required data and changing to named tuples

In [2]:
# Loading in required numpy arrays
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')
base_config = ut.merge_configs(static_configs, runtime_configs)

train_test_dict = ut.load_json5("train_test_dict")
bin_metric_dict = ut.load_json5("bin_metric_dict")

user_counts = ut.load_data("user_counts", "df")
user_interactions = ut.load_data("user_interactions", "df")
user_mapping = ut.load_data("user_mapping", "df")

degen_mask = ut.load_data("degen_mask", "np")
interpolation_weights = ut.load_data("interpolation_weights", "np")

quadratic_interpolation = False

# Loading initial grids
u_init = ut.load_data("u_init", "np")
v_init = ut.load_data("v_init", "np")

p_init = ut.load_data("p_init", "np")
u_pos_init = ut.load_data("u_pos_init", "np")
v_pos_init = ut.load_data("v_pos_init", "np")

n_counts_init = ut.load_data("n_counts_init", "np")

u_clustering = ut.load_data("u_clustering", "np")
v_clustering = ut.load_data("v_clustering", "np")

u_pos_clustering = ut.load_data("u_pos_clustering", "np")
v_pos_clustering = ut.load_data("v_pos_clustering", "np")
p_pos_clustering = ut.load_data("p_pos_clustering", "np")


### Converting to named tuples

In [3]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts,)
output_idx_nt, model_idx_nt = (b.get_model_and_output_idx_nt())
train_test_nt_class = b.dictionary_to_named_tuple_class('train_test_nt',train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)
bin_metric_nt = b.dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating tuner class for runs

In [4]:
# Creating user types and tuner
user_type_groups = (user_mapping.sort('user_id')['source_user_type'].to_numpy() == 'machine').astype(np.int8)
t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class, user_type_groups)

# Validation Runs

### Unsmoothed runner

In [5]:
hyperparams = ut.load_json5('hyper_choices')
experiment_name = 'LL_runner'
hurdle_model = True
results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model,  hyperparams=hyperparams, 
                    train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/8 in 69.9s
finished_config 2/8 in 52.1s
finished_config 3/8 in 51.6s
finished_config 4/8 in 50.9s
finished_config 5/8 in 51.3s
finished_config 6/8 in 50.9s
finished_config 7/8 in 51.8s
finished_config 8/8 in 51.8s


#### Cluster Smoothing Runner

In [5]:
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
hyperparams = ut.load_json5('hyper_choices')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

In [ ]:
experiment_name = 'cluster_smoothing'

cluster_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/100 in 70.3s
finished_config 2/100 in 65.9s
finished_config 3/100 in 53.4s
finished_config 4/100 in 53.2s
finished_config 5/100 in 51.7s
finished_config 6/100 in 52.1s
finished_config 7/100 in 51.3s
finished_config 8/100 in 51.4s
finished_config 9/100 in 51.3s
finished_config 10/100 in 51.4s
finished_config 11/100 in 51.4s
finished_config 12/100 in 51.4s
finished_config 13/100 in 51.3s
finished_config 14/100 in 51.3s
finished_config 15/100 in 51.5s
finished_config 16/100 in 51.7s
finished_config 17/100 in 51.5s
finished_config 18/100 in 51.3s
finished_config 19/100 in 51.4s
finished_config 20/100 in 51.6s
finished_config 21/100 in 51.3s
finished_config 22/100 in 51.4s
finished_config 23/100 in 51.4s
finished_config 24/100 in 51.4s
finished_config 25/100 in 51.9s
finished_config 26/100 in 52.0s
finished_config 27/100 in 51.1s
finished_config 28/100 in 51.3s
finished_config 29/100 in 51.4s
finished_config 30/100 in 51.8s
finished_config 31/100 in 51.3s
finished_config 3

#### Global Smoothing Runner

In [5]:
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
hyperparams = ut.load_json5('hyper_choices')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

In [6]:
experiment_name = 'global_smoothing'

global_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/100 in 83.9s
finished_config 2/100 in 66.2s
finished_config 3/100 in 66.1s
finished_config 4/100 in 66.5s
finished_config 5/100 in 66.1s
finished_config 6/100 in 66.4s
finished_config 7/100 in 65.9s
finished_config 8/100 in 65.5s
finished_config 9/100 in 65.7s
finished_config 10/100 in 65.7s
finished_config 11/100 in 66.0s
finished_config 12/100 in 65.7s
finished_config 13/100 in 65.5s
finished_config 14/100 in 65.4s
finished_config 15/100 in 64.8s
finished_config 16/100 in 64.6s
finished_config 17/100 in 64.9s
finished_config 18/100 in 64.6s
finished_config 19/100 in 63.9s
finished_config 20/100 in 62.6s
finished_config 21/100 in 60.5s
finished_config 22/100 in 60.2s
finished_config 23/100 in 58.9s
finished_config 24/100 in 59.2s
finished_config 25/100 in 58.0s
finished_config 26/100 in 58.2s
finished_config 27/100 in 57.8s
finished_config 28/100 in 57.5s
finished_config 29/100 in 56.9s
finished_config 30/100 in 56.9s
finished_config 31/100 in 56.8s
finished_config 3

# Test final run

#### Lambert Liu Unsmoothed runner

In [5]:
experiment_name = 'no_smoothing'

best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

t.run_test_seeds(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, selected_config=best_models[experiment_name], 
    train_test_dict=train_test_dict, base_config=base_config, degen_mask=degen_mask, bin_metric_dict=bin_metric_dict)

#### Cluster Smoothing Runner

In [5]:
experiment_name = 'cluster_smoothing'

best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

t.run_test_seeds(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, selected_config=best_models[experiment_name], 
    train_test_dict=train_test_dict, base_config=base_config, degen_mask=degen_mask, bin_metric_dict=bin_metric_dict)

finished_seed 1/100 in 123.7s
finished_seed 2/100 in 119.8s
finished_seed 3/100 in 119.5s
finished_seed 4/100 in 119.0s
finished_seed 5/100 in 118.2s
finished_seed 6/100 in 118.5s
finished_seed 7/100 in 118.7s
finished_seed 8/100 in 118.9s
finished_seed 9/100 in 117.5s
finished_seed 10/100 in 117.1s
finished_seed 11/100 in 114.5s
finished_seed 12/100 in 112.4s
finished_seed 13/100 in 111.1s
finished_seed 14/100 in 111.7s
finished_seed 15/100 in 110.7s
finished_seed 16/100 in 110.2s
finished_seed 17/100 in 110.1s
finished_seed 18/100 in 108.3s
finished_seed 19/100 in 107.7s
finished_seed 20/100 in 107.6s
finished_seed 21/100 in 107.6s
finished_seed 22/100 in 106.6s
finished_seed 23/100 in 106.8s
finished_seed 24/100 in 105.0s
finished_seed 25/100 in 105.8s
finished_seed 26/100 in 104.9s
finished_seed 27/100 in 106.3s
finished_seed 28/100 in 104.5s
finished_seed 29/100 in 106.4s
finished_seed 30/100 in 105.9s
finished_seed 31/100 in 106.0s
finished_seed 32/100 in 104.6s
finished_seed 33/

#### Global smoothing runner

In [5]:
experiment_name = 'global_smoothing'

best_models = ut.load_json5('best_configs')
hurdle_nb_model = ut.load_json5('hurdle_nb_model')

t.run_test_seeds(experiment_name=experiment_name, hurdle_nb_model=hurdle_nb_model, selected_config=best_models[experiment_name], 
    train_test_dict=train_test_dict, base_config=base_config, degen_mask=degen_mask, bin_metric_dict=bin_metric_dict)

finished_seed 1/100 in 105.3s
finished_seed 2/100 in 100.8s
finished_seed 3/100 in 107.1s
finished_seed 4/100 in 111.8s
finished_seed 5/100 in 114.8s
finished_seed 6/100 in 112.5s
finished_seed 7/100 in 108.4s
finished_seed 8/100 in 105.8s
finished_seed 9/100 in 102.4s
finished_seed 10/100 in 100.2s
finished_seed 11/100 in 100.3s
finished_seed 12/100 in 100.1s
finished_seed 13/100 in 101.0s
finished_seed 14/100 in 101.1s
finished_seed 15/100 in 101.4s
finished_seed 16/100 in 101.5s
finished_seed 17/100 in 101.5s
finished_seed 18/100 in 101.2s
finished_seed 19/100 in 101.7s
finished_seed 20/100 in 104.4s
finished_seed 21/100 in 101.6s
finished_seed 22/100 in 100.7s
finished_seed 23/100 in 102.0s
finished_seed 24/100 in 99.2s
finished_seed 25/100 in 100.6s
finished_seed 26/100 in 100.6s
finished_seed 27/100 in 100.3s
finished_seed 28/100 in 99.8s
finished_seed 29/100 in 100.0s
finished_seed 30/100 in 100.4s
finished_seed 31/100 in 99.1s
finished_seed 32/100 in 100.4s
finished_seed 33/100